# Clean the GWAS association table

This notebook:

1. reads the GWAS Catalog TSV file;
2. keeps a concise set of useful association columns;
3. expands `MAPPED_GENE` so that each row contains only one mapped gene, while all other association columns are copied;
4. writes a second table containing only `MAPPED_GENE` and `SNPS`.

Multiple genes are split when they are separated by a comma, semicolon, or a **spaced** dash. A dash inside a gene symbol, such as `CR1-AS1`, is preserved.

## 1. Load packages

In [ ]:
required_packages <- c("readr", "dplyr", "tidyr", "stringr")

missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0) {
  stop(
    "Missing R packages: ", paste(missing_packages, collapse = ", "),
    "\nInstall them with: install.packages(c(",
    paste(sprintf('"%s"', missing_packages), collapse = ", "),
    "))"
  )
}

suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(tidyr)
  library(stringr)
})

options(stringsAsFactors = FALSE)

## 2. Set the input file

The notebook first checks the current working directory and then `/mnt/data`. Edit `INPUT_FILE_CANDIDATES` when the file is stored elsewhere.

In [ ]:
input_basenames <- c(
  "gwas-association-downloaded_2026-08-28-accessionId_GCST90027158(1).tsv",
  "gwas-association-downloaded_2026-08-28-accessionId_GCST90027158.tsv"
)

INPUT_FILE_CANDIDATES <- c(
  file.path(getwd(), input_basenames),
  file.path("/mnt/data", input_basenames)
)

existing_files <- INPUT_FILE_CANDIDATES[file.exists(INPUT_FILE_CANDIDATES)]

if (length(existing_files) == 0) {
  stop(
    "Input GWAS file was not found. Paths checked:\n",
    paste(INPUT_FILE_CANDIDATES, collapse = "\n")
  )
}

INPUT_FILE <- normalizePath(existing_files[[1]], mustWork = TRUE)
OUTDIR <- file.path(dirname(INPUT_FILE), "GWAS_cleaned_tables")
dir.create(OUTDIR, recursive = TRUE, showWarnings = FALSE)

cat("Input file:", INPUT_FILE, "\n")
cat("Output directory:", OUTDIR, "\n")

## 3. Read the GWAS table

In [ ]:
gwas_raw <- read_tsv(
  INPUT_FILE,
  col_types = cols(.default = col_character()),
  show_col_types = FALSE,
  progress = FALSE,
  trim_ws = TRUE,
  name_repair = "minimal"
)

required_columns <- c("MAPPED_GENE", "SNPS")
missing_required <- setdiff(required_columns, names(gwas_raw))

if (length(missing_required) > 0) {
  stop(
    "The input table is missing required columns: ",
    paste(missing_required, collapse = ", ")
  )
}

cat("Original rows:", nrow(gwas_raw), "\n")
cat("Original columns:", ncol(gwas_raw), "\n\n")
cat("Column names:\n")
print(names(gwas_raw))

head(gwas_raw, 5)

## 4. Keep useful columns and place one mapped gene in each row

`USEFUL_COLUMNS` can be edited if a different subset is needed. Optional columns that are absent from the source table are skipped automatically; `MAPPED_GENE` and `SNPS` are required.

In [ ]:
USEFUL_COLUMNS <- c(
  "CHR_ID",
  "CHR_POS",
  "REGION",
  "SNPS",
  "STRONGEST SNP-RISK ALLELE",
  "MAPPED_GENE",
  "REPORTED GENE(S)",
  "CONTEXT",
  "RISK ALLELE FREQUENCY",
  "P-VALUE",
  "PVALUE_MLOG",
  "OR or BETA",
  "95% CI (TEXT)",
  "MAPPED_TRAIT"
)

available_useful_columns <- USEFUL_COLUMNS[USEFUL_COLUMNS %in% names(gwas_raw)]
missing_optional_columns <- setdiff(USEFUL_COLUMNS, names(gwas_raw))

cat("Columns retained:\n")
print(available_useful_columns)

if (length(missing_optional_columns) > 0) {
  cat("\nOptional columns not present and therefore skipped:\n")
  print(missing_optional_columns)
}

gwas_one_gene_per_row <- gwas_raw %>%
  select(all_of(available_useful_columns)) %>%
  mutate(MAPPED_GENE = str_squish(as.character(MAPPED_GENE))) %>%
  separate_rows(
    MAPPED_GENE,
    sep = "\\s*[,;]\\s*|\\s+-\\s+"
  ) %>%
  mutate(MAPPED_GENE = str_trim(MAPPED_GENE)) %>%
  filter(!is.na(MAPPED_GENE), MAPPED_GENE != "")

cat("\nRows after expanding MAPPED_GENE:", nrow(gwas_one_gene_per_row), "\n")
cat(
  "Unique mapped genes:",
  n_distinct(gwas_one_gene_per_row$MAPPED_GENE),
  "\n"
)

head(gwas_one_gene_per_row, 20)

## 5. Save the expanded useful-column table

In [ ]:
USEFUL_OUTPUT <- file.path(
  OUTDIR,
  "GCST90027158_useful_columns_one_gene_per_row.tsv"
)

write_tsv(
  gwas_one_gene_per_row,
  USEFUL_OUTPUT,
  na = ""
)

cat("Saved:", USEFUL_OUTPUT, "\n")

## 6. Create and save the `MAPPED_GENE`–`SNPS` table

This table follows the expanded table above, so an association with several mapped genes contributes one gene–SNP row per gene. The original row order is retained. Uncomment `distinct()` when only unique gene–SNP pairs are desired.

In [ ]:
mapped_gene_snps <- gwas_one_gene_per_row %>%
  select(MAPPED_GENE, SNPS)
  # %>% distinct()  # Optional: keep only unique gene-SNP pairs

GENE_SNP_OUTPUT <- file.path(
  OUTDIR,
  "GCST90027158_MAPPED_GENE_SNPS.tsv"
)

write_tsv(
  mapped_gene_snps,
  GENE_SNP_OUTPUT,
  na = ""
)

cat("Rows in MAPPED_GENE-SNPS table:", nrow(mapped_gene_snps), "\n")
cat("Saved:", GENE_SNP_OUTPUT, "\n")

head(mapped_gene_snps, 20)

## 7. Output summary

In [ ]:
output_summary <- tibble(
  table = c(
    "Useful columns; one mapped gene per row",
    "MAPPED_GENE and SNPS only"
  ),
  rows = c(
    nrow(gwas_one_gene_per_row),
    nrow(mapped_gene_snps)
  ),
  output_file = c(
    USEFUL_OUTPUT,
    GENE_SNP_OUTPUT
  )
)

output_summary